In [2]:
!pwd

/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/drl-training


In [3]:
import joblib
import numpy as np
import xgboost as xgb

# Load everything in your environment wrapper
X_scaler = joblib.load("../ml_training/X_scaler.pkl")
mu3_scaler = joblib.load("../ml_training/y_scaler.pkl")
xgb_model = xgb.XGBRegressor()
xgb_model.load("../ml_training/xgb.ubj")


def step_with_sl_prediction(raw_state, base_env):
    # 1. Extract the raw, unnormalized partial features available right now
    # (Shape should match the X_raw features exactly)
    mu0, mu1, mu2, mu3, C, CV, Ln = raw_state[0:7]
    CV_SP = base_env.SP['CV'][base_env.t] if hasattr(base_env, 't') else 1.0
    Ln_SP = base_env.SP['Ln'][base_env.t] if hasattr(base_env, 't') else 15.0
    
    raw_features = np.array([mu0, mu1, mu2, C, CV, Ln, CV_SP, Ln_SP]).reshape(1, -1)

    # 2. Scale features using the input-specific scaler
    X_scaled = X_scaler.transform(raw_features)

    # 3. Predict the normalized mu3
    pred_normalized_mu3 = xgb_model.predict(X_scaled).reshape(-1, 1)

    # 4. Safely inverse-transform ONLY mu3 back to its real-world physical value
    # Because mu3_scaler only knows about mu3, this will never cause a dimension mismatch.
    actual_mu3_value = mu3_scaler.inverse_transform(pred_normalized_mu3)[0][0]

    # 5. Hand the true unnormalized mu3 value over to the RL environment
    # The pc-gym environment wrapper takes over from here and normalizes it internally.
    full_observation = env_partial_obs.inject_mu3(actual_mu3_value)

    return full_observation

FileNotFoundError: [Errno 2] No such file or directory: '../ml_training/X_scaler.pkl'

In [2]:
import numpy as np
from pcgym import make_env

# =====================================================================
# UPDATED TIMELINE PARAMETERS
# =====================================================================
T = 30.0       # Total time horizon = 30 min
nsteps = 30    # N = 30 discrete time steps (resulting in dt = 1.0 min)

# =====================================================================
# TARGET SETPOINTS (SP) FROM TABLE 2
# =====================================================================
SP = {
    'CV': [1.00 for _ in range(nsteps)],
    'Ln': [15.00 for _ in range(nsteps)]
}

# =====================================================================
# UPDATED ACTION SPACE (Incremental control bounds)
# =====================================================================
action_space = {
    'low': np.array([-1.0]),   # Minimum delta T step allowed per minute
    'high': np.array([1.0])    # Maximum delta T step allowed per minute
}

# =====================================================================
# OBSERVATION SPACE (State Bounds from Table 2 + Setpoints)
# =====================================================================
# Order: [mu0, mu1, mu2, mu3, C, CV, Ln, CV_SP, Ln_SP]
observation_space = {
    'low' : np.array([0.0, 0.0, 0.0, 0.0, 0.00, 0.00, 0.00, 0.00, 0.00]),
    'high' : np.array([1.0e20, 1.0e20, 1.0e20, 1.0e20, 0.50, 2.00, 20.00, 2.00, 20.00])  
}

# =====================================================================
# ENVIRONMENT PARAMETERS SETUP
# =====================================================================
env_params = {
    'N': nsteps, 
    'tsim': T, 
    'SP': SP, 
    'o_space': observation_space, 
    'a_space': action_space,
    
    # Initial conditions (x0) including the initial setpoints
    'x0': np.array([1.50e3, 2.30e4, 1.80e6, 2.50e8, 0.16, 1.00, 15.00, 1.00, 15.00]),
    
    'r_scale': {
        'CV': 1e1,
        'Ln': 1e0
    },
    
    'model': 'crystallization', 
    
    'normalise_a': True, 
    'normalise_o': True, 
    'noise': True, 
    'integration_method': 'jax', 
    'noise_percentage': 0.01, 
}
# env_params['custom_reward'] = oracle_reward
# Initialize Environment
env = make_env(env_params)
obs, info = env.reset()

print("Environment successfully setted!")

Environment successfully setted!


/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:231: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/Users/phattharapongduangkham/Documents/crystallization-partial-drl-ml/.venv/lib/python3.12/site-packages/gymnasium/spaces/box.py:297: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [3]:
import gymnasium as gym
import joblib
import numpy as np

class CrystalObservationWrapper(gym.Wrapper):

    def __init__(self, env, x_scaler_path, mu3_scaler_path, xgb_model_path):
        super().__init__(env)
        # 1. Load your separate SL scalers and XGBoost model [cite: 13, 33]
        self.X_scaler = joblib.load(x_scaler_path)
        self.mu3_scaler = joblib.load(mu3_scaler_path)

        import xgboost as xgb

        self.xgb_model = xgb.Booster()
        self.xgb_model.load_model("../ml-training/xgb_model.ubj")
        # self.xgb_model = joblib.load(xgb_model_path)

    def step(self, action):
        # 2. Step the actual environment using the agent's action
        # This returns the raw/partial observation from pc-gym
        obs, reward, terminated, truncated, info = self.env.step(action)

        # 3. Apply the step_with_sl_prediction logic to fix the observation
        modified_obs = self._predict_and_inject_mu3(obs)

        return modified_obs, reward, terminated, truncated, info

    def reset(self, **kwargs):
        # Must also modify the initial observation when resetting the environment
        obs, info = self.env.reset(**kwargs)
        modified_obs = self._predict_and_inject_mu3(obs)
        return modified_obs, info

    def _predict_and_inject_mu3(self, partial_obs):
        # Extract features required by XGBoost (ensure unnormalized state) [cite: 3, 22]
        # NOTE: Adapt this line depending on how pc-gym structured your observation array
        raw_features = self._extract_features_from_obs(partial_obs)

        # Reshape to 2D array for sklearn [cite: 28]
        raw_features = np.array(raw_features).reshape(1, -1)

        # Scale features using the input-specific scaler [cite: 33]
        X_scaled = self.X_scaler.transform(raw_features)

        # Predict the normalized mu3 [cite: 4]
        pred_normalized_mu3 = self.xgb_model.predict(X_scaled).reshape(-1, 1)

        # Inverse-transform ONLY mu3 back to its real physical value [cite: 22, 33]
        actual_mu3_value = self.mu3_scaler.inverse_transform(pred_normalized_mu3)[0][0]

        # Inject the real-world mu3 into the environment's observation [cite: 22]
        # pc-gym will then handle its own internal normalization for the PPO policy 
        full_obs = self._inject_into_obs_array(partial_obs, actual_mu3_value)

        return full_obs

    def _extract_features_from_obs(self, obs):
        # Helper: Extract your XGBoost input variables out of the observation array
        # Example: return obs[:-1] if mu3 was the last element
        return obs

    def _inject_into_obs_array(self, obs, mu3_val):
        # Helper: Place the real mu3 value back into the correct observation slot
        # Example: obs[-1] = mu3_val
        return obs

In [ ]:
from stable_baselines3 import PPO

# Wrap it with your SL adaptation layer
env = CrystalObservationWrapper(
    env=env,
    x_scaler_path="../ml-training/X_scaler.pkl",
    mu3_scaler_path="../ml-training/y_scaler.pkl",
    xgb_model_path="../ml-training/xgb_model.ubj",
)

# Train PPO normally. It will seamlessly receive the estimated mu3 values [cite: 1, 5]
model = PPO("MlpPolicy", env)
model.learn(total_timesteps=30000)  # Like your expert policy [cite: 2]

: 